<a href="https://colab.research.google.com/github/kkinca/medical-rag-assistant/blob/main/RAG_Project_Notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Problem Statement

### Business Context

The healthcare industry is rapidly evolving, with professionals facing increasing challenges in managing vast volumes of medical data while delivering accurate and timely diagnoses. The need for quick access to comprehensive, reliable, and up-to-date medical knowledge is critical for improving patient outcomes and ensuring informed decision-making in a fast-paced environment.

Healthcare professionals often encounter information overload, struggling to sift through extensive research and data to create accurate diagnoses and treatment plans. This challenge is amplified by the need for efficiency, particularly in emergencies, where time-sensitive decisions are vital. Furthermore, access to trusted, current medical information from renowned manuals and research papers is essential for maintaining high standards of care.

To address these challenges, healthcare centers can focus on integrating systems that streamline access to medical knowledge, provide tools to support quick decision-making, and enhance efficiency. Leveraging centralized knowledge platforms and ensuring healthcare providers have continuous access to reliable resources can significantly improve patient care and operational effectiveness.

**Common Questions to Answer**

**1. Diagnostic Assistance**: "What are the common symptoms and treatments for pulmonary embolism?"

**2. Drug Information**: "Can you provide the trade names of medications used for treating hypertension?"

**3. Treatment Plans**: "What are the first-line options and alternatives for managing rheumatoid arthritis?"

**4. Specialty Knowledge**: "What are the diagnostic steps for suspected endocrine disorders?"

**5. Critical Care Protocols**: "What is the protocol for managing sepsis in a critical care unit?"

### Objective

As an AI specialist, your task is to develop a RAG-based AI solution using renowned medical manuals to address healthcare challenges. The objective is to **understand** issues like information overload, **apply** AI techniques to streamline decision-making, **analyze** its impact on diagnostics and patient outcomes, **evaluate** its potential to standardize care practices, and **create** a functional prototype demonstrating its feasibility and effectiveness.

### Data Description

The **Merck Manuals** are medical references published by the American pharmaceutical company Merck & Co., that cover a wide range of medical topics, including disorders, tests, diagnoses, and drugs. The manuals have been published since 1899, when Merck & Co. was still a subsidiary of the German company Merck.

The manual is provided as a PDF with over 4,000 pages divided into 23 sections.

## Installing and Importing Necessary Libraries and Dependencies

In [ ]:
# ==============================
# 1. INSTALL REQUIRED PACKAGES
# ==============================

!pip install -q --upgrade pip

!pip install -q \
    langchain==0.3.7 \
    langchain-community==0.3.7 \
    langchain-core==0.3.17 \
    langchain-text-splitters==0.3.2 \
    chromadb==0.5.5 \
    sentence-transformers==3.0.1 \
    pypdf \
    llama-cpp-python==0.2.90

In [ ]:
print("Install complete.")
print("IMPORTANT: Now go to Runtime > Restart session, then continue running the notebook from the next cell.")

In [ ]:
# ==============================
# 2. IMPORT LIBRARIES
# ==============================

import os
import json
import warnings

warnings.filterwarnings("ignore")

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_community.llms import LlamaCpp
from langchain.prompts import PromptTemplate
from langchain.chains import RetrievalQA
from langchain.embeddings.base import Embeddings

from sentence_transformers import SentenceTransformer

In [ ]:
# ==============================
# 3. FILE PATHS
# ==============================

PDF_PATH = "/content/medical_diagnosis_manual.pdf"
CHROMA_DIR = "medical_db"

print("PDF exists:", os.path.exists(PDF_PATH))

In [ ]:
# ==============================
# 4. LOAD PDF
# ==============================

loader = PyPDFLoader(PDF_PATH)
documents = loader.load()

print(f"Loaded {len(documents)} pages from the medical manual.")

In [ ]:
# ==============================
# 5. SPLIT TEXT INTO CHUNKS
# ==============================

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

docs = text_splitter.split_documents(documents)

print(f"Created {len(docs)} chunks.")

In [ ]:
# ==============================
# 6. EMBEDDING MODEL
# ==============================

class SentenceTransformerEmbeddings(Embeddings):
    def __init__(self, model_name="all-MiniLM-L6-v2"):
        self.model = SentenceTransformer(model_name)

    def embed_documents(self, texts):
        return self.model.encode(texts).tolist()

    def embed_query(self, text):
        return self.model.encode(text).tolist()

embedding_model = SentenceTransformerEmbeddings("all-MiniLM-L6-v2")

print("Embedding model loaded.")

In [ ]:
# ==============================
# 7. CREATE VECTOR DATABASE
# ==============================

import os
os.environ["ANONYMIZED_TELEMETRY"] = "False"

vectordb = Chroma.from_documents(
    documents=docs,
    embedding=embedding_model,
    persist_directory="medical_db"
)

retriever = vectordb.as_retriever(search_kwargs={"k": 3})

print("Vector database created and retriever ready.")

In [ ]:
# ==============================
# 8A. DOWNLOAD THE GGUF MODEL
# ==============================

from huggingface_hub import hf_hub_download

model_path = hf_hub_download(
    repo_id="TheBloke/Mistral-7B-Instruct-v0.2-GGUF",
    filename="mistral-7b-instruct-v0.2.Q6_K.gguf"
)

print("Model downloaded to:", model_path)

In [ ]:
# ==============================
# 8. LOAD LOCAL LLM
# ==============================

llm = LlamaCpp(
    model_path=model_path,
    temperature=0.2,
    max_tokens=350,
    top_p=0.9,
    top_k=40,
    n_ctx=4096,
    verbose=False
)

print("LLM loaded.")

In [ ]:
# ==============================
# 9. LLM-ONLY BASELINE
# ==============================

baseline_prompt = PromptTemplate(
    input_variables=["question"],
    template="""
You are a clinical medical assistant.
Provide structured, step-by-step, medically accurate responses.
Avoid speculation.
Clearly distinguish symptoms, diagnosis, and treatment.
If uncertain, state limitations explicitly.

Question: {question}

Answer:
"""
)

def run_llm_baseline(question):
    prompt = baseline_prompt.format(question=question)
    return llm.invoke(prompt)

In [ ]:
question = "What is the recommended management for sepsis?"
baseline_answer = run_llm_baseline(question)
print(baseline_answer)

In [ ]:
# ==============================
# 10. RAG CHAIN
# ==============================

rag_prompt = PromptTemplate(
    input_variables=["context", "question"],
    template="""
You are a clinical medical assistant.
Answer the question using only the provided context from the Merck Manual.
If the information is not available in the context, clearly state that it is not found.

Context:
{context}

Question:
{question}

Provide a structured, medically accurate answer based strictly on the context.
"""
)

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    chain_type="stuff",
    chain_type_kwargs={"prompt": rag_prompt},
    return_source_documents=True
)

In [ ]:
question = "What is the recommended management for sepsis?"
rag_result = qa_chain.invoke({"query": question})

print("RAG ANSWER:\n")
print(rag_result["result"])

In [ ]:
print("\nRETRIEVED SOURCES:\n")

for i, doc in enumerate(rag_result["source_documents"], 1):
    print(f"--- Source {i} ---")
    print(doc.page_content[:1000])
    print("\n")

In [ ]:
# ==============================
# 11. SAMPLE QUESTIONS
# ==============================

questions = [
    "What is the recommended management for sepsis?",
    "What are the symptoms and treatment options for appendicitis?",
    "What causes sudden patchy hair loss and how is it treated?",
    "How is traumatic brain injury managed?",
    "What precautions and recovery steps are recommended for a leg fracture?"
]

for q in questions:
    result = qa_chain.invoke({"query": q})
    print("=" * 80)
    print("QUESTION:", q)
    print("\nANSWER:\n", result["result"])
    print("\n")

In [ ]:
# ==============================
# 12. EVALUATION PROMPTS
# ==============================

groundedness_prompt = """
Evaluate whether the answer is fully supported by the provided context.

Score from 1 to 5:
1 = Not supported by context
3 = Partially supported
5 = Fully grounded in context

Provide structured reasoning for the assigned score.

Question: {question}

Context:
{context}

Answer:
{answer}
"""

relevance_prompt = """
Evaluate how directly and completely the answer addresses the user's question.

Score from 1 to 5:
1 = Not relevant
3 = Partially relevant
5 = Fully relevant and complete

Provide structured reasoning for the assigned score.

Question: {question}

Answer:
{answer}
"""

## Question Answering using LLM

### Response

In [ ]:
def response(query,max_tokens=128,temperature=0,top_p=0.95,top_k=50):
    model_output = llm(
      prompt=query,
      max_tokens=max_tokens,
      temperature=temperature,
      top_p=top_p,
      top_k=top_k
    )

    return model_output['choices'][0]['text']

In [ ]:
question = "What treatment options are available for managing hypertension?"
baseline_answer = run_llm_baseline(question)
print(baseline_answer)

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [ ]:
question = "What is the protocol for managing sepsis in a critical care unit?"
baseline_answer = run_llm_baseline(question)
print(baseline_answer)

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [ ]:
# ==============================
# BASELINE TEST QUESTION 2
# ==============================

question = "What are the common symptoms of appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"
baseline_answer = run_llm_baseline(question)

print("QUESTION:\n", question)
print("\nBASELINE ANSWER:\n")
print(baseline_answer)

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [ ]:
# ==============================
# BASELINE TEST QUESTION 3
# ==============================

question = "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"
baseline_answer = run_llm_baseline(question)

print("QUESTION:\n", question)
print("\nBASELINE ANSWER:\n")
print(baseline_answer)

### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [ ]:
# ==============================
# BASELINE TEST QUESTION 4
# ==============================

question = "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"
baseline_answer = run_llm_baseline(question)

print("QUESTION:\n", question)
print("\nBASELINE ANSWER:\n")
print(baseline_answer)

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [ ]:
# ==============================
# BASELINE TEST QUESTION 5
# ==============================

question = "What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?"
baseline_answer = run_llm_baseline(question)

print("QUESTION:\n", question)
print("\nBASELINE ANSWER:\n")
print(baseline_answer)

## Question Answering using LLM with Prompt Engineering

In [ ]:
system_prompt = "You are a knowledgeable medical assistant. Provide clear, structured, and professional answers based on medical knowledge. Keep responses informative and concise."

# **Prompt Engineering – Parameter Tuning Experiments**

## Prompt Engineering – Parameter Tuning

Different parameter settings were tested to improve structure, completeness, and clinical consistency in LLM-only responses.

The final configuration selected was:

- Temperature = 0.2
- Max Tokens = 350
- Top-p = 0.9
- Top-k = 40

This setting provided the best balance of step-by-step clarity, response completeness, and reduced variability.

In [ ]:
# ==============================
# PROMPT ENGINEERING RESULTS
# ==============================

prompt_tuning_results = [
    {"Run": 1, "Temperature": 0.0, "MaxTokens": 250, "Top-p": 0.90, "Top-k": 40, "Outcome": "Balanced length; Stepwise; More assertive"},
    {"Run": 2, "Temperature": 0.3, "MaxTokens": 350, "Top-p": 0.95, "Top-k": 50, "Outcome": "Balanced length; Stepwise; Cautious"},
    {"Run": 3, "Temperature": 0.2, "MaxTokens": 350, "Top-p": 0.90, "Top-k": 40, "Outcome": "Balanced length; Stepwise; Cautious"},
    {"Run": 4, "Temperature": 0.6, "MaxTokens": 300, "Top-p": 0.95, "Top-k": 50, "Outcome": "Balanced length; Stepwise; More assertive"},
    {"Run": 5, "Temperature": 0.2, "MaxTokens": 200, "Top-p": 0.85, "Top-k": 30, "Outcome": "Balanced length; Stepwise; More assertive"},
]

for row in prompt_tuning_results:
    print(row)

## Data Preparation for RAG

In [ ]:
## Data Preparation for RAG

The Merck Manual PDF was loaded, split into 1000-character chunks with 200-character overlap, embedded using `all-MiniLM-L6-v2`, and stored in a Chroma vector database for similarity-based retrieval.

### Loading the Data

In [ ]:
# ==============================
# LOAD PDF
# ==============================

PDF_PATH = "/content/medical_diagnosis_manual.pdf"

loader = PyPDFLoader(PDF_PATH)
documents = loader.load()

print(f"Loaded {len(documents)} pages from the medical manual.")

### Data Chunking

In [ ]:
# ==============================
# SPLIT TEXT INTO CHUNKS
# ==============================

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

docs = text_splitter.split_documents(documents)

print(f"Created {len(docs)} chunks.")

In [ ]:
# Preview one chunk
print(docs[0].page_content[:1000])

### Embedding

In [ ]:
# ==============================
# EMBEDDING MODEL
# ==============================

class SentenceTransformerEmbeddings(Embeddings):
    def __init__(self, model_name="all-MiniLM-L6-v2"):
        self.model = SentenceTransformer(model_name)

    def embed_documents(self, texts):
        return self.model.encode(texts).tolist()

    def embed_query(self, text):
        return self.model.encode(text).tolist()

embedding_model = SentenceTransformerEmbeddings("all-MiniLM-L6-v2")

print("Embedding model loaded.")

### Vector Database

In [ ]:
# ==============================
# CREATE VECTOR DATABASE
# ==============================

vectordb = Chroma.from_documents(
    documents=docs,
    embedding=embedding_model,
    persist_directory="medical_db"
)

retriever = vectordb.as_retriever(search_kwargs={"k": 3})

print("Vector database created and retriever ready.")

### Retriever

In [ ]:
# Quick retriever test
test_docs = retriever.invoke("What is sepsis?")
print(f"Retrieved {len(test_docs)} documents.")
print(test_docs[0].page_content[:1000])

### System and User Prompt Template

Prompts guide the model to generate accurate responses. Here, we define two parts:

    1. The system message describing the assistant's role.
    2. A user message template including context and the question.

In [ ]:
# ==============================
# RAG PROMPT TEMPLATE
# ==============================

rag_prompt = PromptTemplate(
    input_variables=["context", "question"],
    template="""
You are a clinical medical assistant.
Answer the question using only the provided context from the Merck Manual.
If the information is not available in the context, clearly state that it is not found.

Context:
{context}

Question:
{question}

Provide a structured, medically accurate answer based strictly on the context.
"""
)

### Response Function

In [ ]:
# ==============================
# RAG CHAIN
# ==============================

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    chain_type="stuff",
    chain_type_kwargs={"prompt": rag_prompt},
    return_source_documents=True
)

print("RAG chain ready.")

## Question Answering using RAG

In [ ]:
# ==============================
# RAG SAMPLE QUESTIONS
# ==============================

questions = [
    "What is the recommended management for sepsis?",
    "What are the symptoms and treatment options for appendicitis?",
    "What causes sudden patchy hair loss and how is it treated?",
    "How is traumatic brain injury managed?",
    "What precautions and recovery steps are recommended for a leg fracture?"
]

for q in questions:
    result = qa_chain.invoke({"query": q})
    print("=" * 80)
    print("QUESTION:", q)
    print("\nANSWER:\n", result["result"])
    print("\n")

## RAG Parameter Tuning

## RAG Parameter Tuning

Different values for `k`, `temperature`, and `max_tokens` were tested to balance completeness, focus, and groundedness. The final configuration selected was:

- `k = 3`
- `temperature = 0.2`
- `max_tokens = 350`

This setting provided the best tradeoff between contextual completeness and reduced redundancy.

In [ ]:
tuning_results = [
    {"Run": 1, "k": 1, "temperature": 0.2, "max_tokens": 300, "Observation": "Insufficient contextual grounding"},
    {"Run": 2, "k": 3, "temperature": 0.2, "max_tokens": 350, "Observation": "Best balance of completeness and focus"},
    {"Run": 3, "k": 5, "temperature": 0.2, "max_tokens": 350, "Observation": "Too much overlap and verbosity"},
    {"Run": 4, "k": 3, "temperature": 0.4, "max_tokens": 350, "Observation": "Slight variability in phrasing"},
    {"Run": 5, "k": 3, "temperature": 0.2, "max_tokens": 250, "Observation": "Response too short and incomplete"},
]

for row in tuning_results:
    print(row)

## Output Evaluation

Let us now use the LLM-as-a-judge method to check the quality of the RAG system on two parameters - retrieval and generation.

- We are using the same Mistral model for evaluation, so basically here the llm is rating itself on how well he has performed in the task.

In [ ]:
# ==============================
# GROUNDEDNESS EVALUATION PROMPT
# ==============================

groundedness_rater_system_message = """
You are a strict medical QA evaluator for a Retrieval-Augmented Generation (RAG) system.
Your job is to judge GROUNDEDNESS only.

Definition:
- Groundedness = how well the answer is supported by the provided context.
- If the answer contains claims that are not present in the context, groundedness drops.
- If the answer contradicts the context, groundedness is very low.

Return ONLY a JSON object with exactly these keys:
{
  "score": integer 1-5,
  "verdict": "<one of: grounded | partially_grounded | not_grounded>",
  "rationale": "<1-3 sentences, cite phrases from the context when possible>"
}

Scoring:
5 = Fully supported by context, no extra claims.
4 = Mostly supported, minor extrapolation.
3 = Some support, but multiple unsupported additions.
2 = Little support, mostly hallucinated.
1 = Not supported or contradicts context.
"""

In [ ]:
# ==============================
# RELEVANCE EVALUATION PROMPT
# ==============================

relevance_rater_system_message = """
You are a strict medical QA evaluator for a Retrieval-Augmented Generation (RAG) system.
Your job is to judge RELEVANCE only.

Definition:
- Relevance = whether the answer directly addresses the question asked.
- Do NOT judge truthfulness unless it affects relevance.

Return ONLY a JSON object with exactly these keys:
{
  "score": integer 1-5,
  "verdict": "<one of: relevant | partially_relevant | not_relevant>",
  "rationale": "<1-3 sentences>"
}

Scoring:
5 = Fully relevant and complete.
4 = Relevant with minor omissions.
3 = Partially relevant.
2 = Weakly relevant.
1 = Not relevant.
"""

## RAG Parameter Tuning

Different values for `k`, `temperature`, and `max_tokens` were tested to balance completeness, focus, and groundedness. The final configuration selected was:

- `k = 3`
- `temperature = 0.2`
- `max_tokens = 350`

This setting provided the best tradeoff between contextual completeness and reduced redundancy.